<a href="https://colab.research.google.com/github/emanuelcatao/analisedadoscpa/blob/main/analise_dados_tcc1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Executa até "Preparação do ambiente", fazer upload de arquivos dentro da pasta "/dados_originais". Seguir com todo o restante


# PARTE 1: SETUP E FUNÇÕES

In [102]:
!pip install -U kaleido

In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
pio.templates.default = "plotly_white"
pio.renderers.default = "colab"

## 1.2. Configurações do Projeto

#### Preparacao do ambiente

In [104]:
import requests
from pathlib import Path

#URL_GITHUB = "https://raw.githubusercontent.com/emanuelcatao/" -- acho que nao posso colocar o xlsx no repo

DADOS_ORIGINAIS = Path("/content/dados_originais")
OUTPUTS_DIR = Path("/content/outputs")
FIGURAS_DIR = OUTPUTS_DIR / "figuras"
HTML_DIR = OUTPUTS_DIR / "html"

FIGURAS_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DADOS_ORIGINAIS.mkdir(parents=True, exist_ok=True)

ARQUIVO_RESPOSTAS = DADOS_ORIGINAIS / "respostas2024.xlsx"

# print("baixando arquivo")
# response = requests.get(URL_GITHUB)
# if response.status_code == 200:
#     with open(ARQUIVO_RESPOSTAS, 'wb') as f:
#         f.write(response.content)
# else:
#     print("erro")

####  Parametros de Categorização

In [105]:
LIKERT_SCALE = [1, 2, 3, 4, 5, 6]
LIKERT_LABELS = {
    1: "Discordo Totalmente",
    2: "Discordo",
    3: "Discordo Parcialmente",
    4: "Concordo Parcialmente",
    5: "Concordo",
    6: "Concordo Totalmente"
}

EIXOS_SINAES = {
    "Eixo 1": {
        "nome": "Planejamento e Avaliação Institucional",
        "questoes": ["q1","q1.1","q2","q2.1","q3","q3.1","q4","q4.1","q5","q6"],
        "cor": "#1f77b4"
    },
    "Eixo 2": {
        "nome": "Desenvolvimento Institucional",
        "questoes": ["q7", "q8", "q9", "q10", "q11", "q12"],
        "cor": "#ff7f0e"
    },
    "Eixo 3": {
        "nome": "Políticas Acadêmicas",
        "questoes": [f"q{i}" for i in range(13, 32)] + ["q29.1"],
        "cor": "#2ca02c"
    },
    "Eixo 4": {
        "nome": "Políticas de Gestão",
        "questoes": [f"q{i}" for i in range(32, 43)],
        "cor": "#d62728"
    },
    "Eixo 5": {
        "nome": "Infraestrutura Física",
        "questoes": [f"q{i}" for i in range(43, 60)],
        "cor": "#9467bd"
    }
}

eixo_name_to_display = {
    EIXOS_SINAES["Eixo 1"]["nome"]: "Planejamento e\nAvaliação",
    EIXOS_SINAES["Eixo 2"]["nome"]: "Desenvolvimento\nInstitucional",
    EIXOS_SINAES["Eixo 3"]["nome"]: "Políticas\nAcadêmicas",
    EIXOS_SINAES["Eixo 4"]["nome"]: "Políticas de\nGestão",
    EIXOS_SINAES["Eixo 5"]["nome"]: "Infraestrutura",
}
ordem_eixos_display = list(eixo_name_to_display.values())

print(EIXOS_SINAES)

CORES_SEGMENTO = {
    "Discente": "#06d6a0",
    "Aluno": "#06d6a0",
    "Egresso": "#ffd166",
    "Docente": "#ef476f",
    "Professor": "#ef476f",
    "Técnico": "#118ab2"
}

print("✅ Configurações carregadas")
print(f"   Arquivo de dados: {ARQUIVO_RESPOSTAS}")
print(f"   Diretório de saída: {FIGURAS_DIR}")


{'Eixo 1': {'nome': 'Planejamento e Avaliação Institucional', 'questoes': ['q1', 'q1.1', 'q2', 'q2.1', 'q3', 'q3.1', 'q4', 'q4.1', 'q5', 'q6'], 'cor': '#1f77b4'}, 'Eixo 2': {'nome': 'Desenvolvimento Institucional', 'questoes': ['q7', 'q8', 'q9', 'q10', 'q11', 'q12'], 'cor': '#ff7f0e'}, 'Eixo 3': {'nome': 'Políticas Acadêmicas', 'questoes': ['q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22', 'q23', 'q24', 'q25', 'q26', 'q27', 'q28', 'q29', 'q30', 'q31', 'q29.1'], 'cor': '#2ca02c'}, 'Eixo 4': {'nome': 'Políticas de Gestão', 'questoes': ['q32', 'q33', 'q34', 'q35', 'q36', 'q37', 'q38', 'q39', 'q40', 'q41', 'q42'], 'cor': '#d62728'}, 'Eixo 5': {'nome': 'Infraestrutura Física', 'questoes': ['q43', 'q44', 'q45', 'q46', 'q47', 'q48', 'q49', 'q50', 'q51', 'q52', 'q53', 'q54', 'q55', 'q56', 'q57', 'q58', 'q59'], 'cor': '#9467bd'}}
✅ Configurações carregadas
   Arquivo de dados: /content/dados_originais/respostas2024.xlsx
   Diretório de saída: /content/outputs/figuras


## 1.3. Funções Auxiliares

In [106]:
# ============================================================================
# FUNÇÕES AUXILIARES
# ============================================================================

def to_likert_numeric(valor):
    """
    Converte valor Likert para numérico (1-6)
    Remove 'NSO', 'Não sei', etc
    """
    if pd.isna(valor):
        return np.nan

    valor_str = str(valor).strip().lower()

    # NSO sempre vira NaN
    if valor_str in ['nso', 'não sei', 'não sei opinar', 'n/a', 'na']:
        return np.nan

    try:
        num = float(valor_str)
        if 1 <= num <= 6:
            return num
    except:
        pass

    return np.nan


def to_binary_numeric(valor):
    """
    NAO SEI SE ISSO É INTERESSANTE OU NÃO
    Converte escala binária S/N para numérico (1=Não, 6=Sim)
    Mantém compatibilidade com escala Likert para cálculos
    NSO vira NaN
    """
    if pd.isna(valor):
        return np.nan

    valor_str = str(valor).strip().lower()

    # NSO sempre vira NaN
    if valor_str in ['nso', 'não sei', 'não sei opinar', 'n/a', 'na']:
        return np.nan

    if valor_str in ['s', 'sim']:
        return 6  # Concordo totalmente
    if valor_str in ['n', 'não', 'nao']:
        return 1  # Discordo totalmente

    return np.nan


def converter_resposta(valor, tipo_escala='likert'):
    """
    Função unificada que detecta ou usa o tipo de escala especificado

    Args:
        valor: resposta a ser convertida
        tipo_escala: 'likert', 'binary' ou 'auto'
    """
    if tipo_escala == 'auto':
        valor_str = str(valor).strip().lower()
        if valor_str in ['s', 'n', 'sim', 'não', 'nao']:
            tipo_escala = 'binary'
        else:
            tipo_escala = 'likert'

    if tipo_escala == 'binary':
        return to_binary_numeric(valor)
    else:
        return to_likert_numeric(valor)

def identificar_segmento(row):
    """
    Identifica o(s) segmento(s) do respondente baseado nas colunas _FORMULARIO
    Retorna lista de segmentos (pode ter múltiplos)
    """
    segmentos = []

    if pd.notna(row.get('ALUNO_FORMULARIO')):
        segmentos.append('Discente')
    if pd.notna(row.get('PROFESSOR_FORMULARIO')):
        segmentos.append('Docente')
    if pd.notna(row.get('TECNICO_FORMULARIO')):
        segmentos.append('Técnico')
    if pd.notna(row.get('EGRESSO_FORMULARIO')):
        segmentos.append('Egresso')

    return segmentos if segmentos else None


def obter_colunas_questoes(df, prefixo='q'):
    """
    Retorna lista de colunas que representam questoes, ex: q1, q1.1, q2, q2.1
    """
    colunas = []
    for col in df.columns:
        if col.startswith(prefixo):
            resto = col[len(prefixo):]
            if resto.replace('.', '').replace('_', '').isdigit():
                colunas.append(col)

    def chave(col):
        parte = col[len(prefixo):].replace('_', '')
        try:
            return float(parte)
        except ValueError:
            return float('inf')

    return sorted(colunas, key=chave)


def estatisticas_descritivas(serie):
    """
    Calcula estatísticas descritivas de uma série.
    """
    serie_limpa = serie.dropna()
    if len(serie_limpa) == 0:
        return {}

    return {
        'n': len(serie_limpa),
        'media': serie_limpa.mean(),
        'mediana': serie_limpa.median(),
        'desvio_padrao': serie_limpa.std(),
        'minimo': serie_limpa.min(),
        'maximo': serie_limpa.max()
    }


def interpretar_correlacao(r):
    """
    Interpreta o coeficiente de correlação (bem simploriamente).
    """
    r_abs = abs(r)

    if r_abs < 0.1:
        forca = "desprezível"
    elif r_abs < 0.3:
        forca = "fraca"
    elif r_abs < 0.5:
        forca = "fraca a moderada"
    elif r_abs < 0.7:
        forca = "moderada"
    elif r_abs < 0.9:
        forca = "forte"
    else:
        forca = "muito forte"

    direcao = "positiva" if r >= 0 else "negativa"
    return f"Correlação {direcao} {forca}"


# Parte 2: Tratamento e visão geral do Dataset

## 2.1. Carregar Dados

In [107]:
try:
    df = pd.read_excel(ARQUIVO_RESPOSTAS, sheet_name="respostas")
except:
    try:
        df = pd.read_excel(ARQUIVO_RESPOSTAS, sheet_name=0)
    except:
        df = pd.read_excel(ARQUIVO_RESPOSTAS)

print(f"   • Respondentes: {len(df):,}")
print(f"   • Variáveis: {len(df.columns)}")
print(f"\nPrimeiras colunas: {list(df.columns[:10])}")

   • Respondentes: 640
   • Variáveis: 84

Primeiras colunas: ['PROFESSOR', 'TECNICO', 'ALUNO', 'EGRESSO', 'EGRESSO_FORMULARIO', 'PROFESSOR_FORMULARIO', 'TECNICO_FORMULARIO', 'ALUNO_FORMULARIO', 'PROFESSOR_CAMPUS', 'PROFESSOR_TITULACAO']


## 2.2. Identificar Estrutura

In [108]:
colunas_questoes = obter_colunas_questoes(df)

print(f"\nEstrutura identificada:")
print(f"   • Questões encontradas: {len(colunas_questoes)}")
print(f"   • Listagem: {colunas_questoes}")


Estrutura identificada:
   • Questões encontradas: 65
   • Listagem: ['q1', 'q1.1', 'q2', 'q2.1', 'q3', 'q3.1', 'q4', 'q4.1', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22', 'q23', 'q24', 'q25', 'q26', 'q27', 'q28', 'q29', 'q29.1', 'q30', 'q31', 'q32', 'q33', 'q34', 'q35', 'q36', 'q37', 'q38', 'q39', 'q40', 'q41', 'q42', 'q43', 'q44', 'q45', 'q46', 'q47', 'q48', 'q49', 'q50', 'q51', 'q52', 'q53', 'q54', 'q55', 'q56', 'q57', 'q58', 'q59', 'q70']


## 2.3. Criar Segmentos

In [109]:
coluna_segmento = None
for possivel in ['SEGMENTO', 'segmento', 'Segmento']:
    if possivel in df.columns:
        coluna_segmento = possivel
        break

if not coluna_segmento:
  df["segmentos"] = df.apply(identificar_segmento, axis=1)
  df = df[df["segmentos"].notna()].copy()

  # respondentes com múltiplos segmentos/grupos
  multi_seg = df[df['segmentos'].apply(lambda x: len(x) > 1)]
  if len(multi_seg) > 0:
      print(f"   ⚠️  {len(multi_seg)} respondentes com múltiplos segmentos")
      print(f"      Expandindo para análise separada por segmento...")

  # uma linha para cada segmento - talvez faça mais sentido deixa apenas um registro, estamos meio que inflando artificialmente aqui
  df = df.explode('segmentos').reset_index(drop=True)

  df = df.rename(columns={'segmentos': 'segmento'})

  coluna_segmento = "segmento"

print(f"\n📊 Distribuição por segmento:")
print(df[coluna_segmento].value_counts())
print(f"\nTotal de registros (linhas): {len(df)}")

   ⚠️  188 respondentes com múltiplos segmentos
      Expandindo para análise separada por segmento...

📊 Distribuição por segmento:
segmento
Discente    464
Egresso     182
Docente     137
Técnico      71
Name: count, dtype: int64

Total de registros (linhas): 854


## 2.4. Panorama Inicial dos Dados


#### Dimensões

In [110]:
print(f"Total de registros: {len(df)}")
print(f"Total de variáveis: {len(df.columns)}")
print(f"Variáveis de resposta (questões): {len(colunas_questoes)}")

Total de registros: 854
Total de variáveis: 85
Variáveis de resposta (questões): 65


#### Distribuicao por Grupo Respondente

In [111]:
segmento_counts = df['segmento'].value_counts()
total_geral = len(df[df['segmento'].notna()])

for seg in ['Discente', 'Docente', 'Técnico', 'Egresso']:
  count = segmento_counts.get(seg, 0)
  pct = (count / total_geral * 100) if total_geral > 0 else 0
  print(f"{seg}: {count} ({pct:.1f}%)")

print(f"\nTotal: {total_geral} respondentes")

Discente: 464 (54.3%)
Docente: 137 (16.0%)
Técnico: 71 (8.3%)
Egresso: 182 (21.3%)

Total: 854 respondentes


#### Distribuicao por Campus e Grupo Respondente

In [112]:
colunas_campus = ['ALUNO_CAMPUS', 'PROFESSOR_CAMPUS', 'TECNICO_CAMPUS', 'EGRESSO_CAMPUS']
colunas_existentes = [col for col in colunas_campus if col in df.columns]

if colunas_existentes:
    df['campus'] = df[colunas_existentes].bfill(axis=1).iloc[:, 0]

    if df['campus'].notna().sum() > 0:
        tabela_campus = pd.crosstab(
            df['campus'],
            df['segmento'],
            margins=True,
            margins_name='TOTAL'
        )

        ordem_colunas = ['Egresso', 'Discente', 'Docente', 'Técnico', 'TOTAL']
        colunas_presentes = [col for col in ordem_colunas if col in tabela_campus.columns]
        tabela_campus = tabela_campus[colunas_presentes]

        tabela_sem_total = tabela_campus[tabela_campus.index != 'TOTAL'].copy()
        tabela_sem_total = tabela_sem_total.sort_values('TOTAL', ascending=False)
        linha_total = tabela_campus[tabela_campus.index == 'TOTAL']
        tabela_campus = pd.concat([tabela_sem_total, linha_total])

        total_geral = tabela_campus.loc['TOTAL', 'TOTAL']
        tabela_campus['%'] = (tabela_campus['TOTAL'] / total_geral * 100).round(1)

        display(tabela_campus)
    else:
        print("⚠️ Nenhum valor encontrado nas colunas de campus.")
else:
    print("⚠️ Nenhuma das colunas de campus existe no DataFrame:")
    print("   ", ", ".join(colunas_campus))


segmento,Egresso,Discente,Docente,Técnico,TOTAL,%
campus,,,,,,
Palmas,68,175,45,28,316,37.0
Arraias,27,138,13,8,186,21.8
Gurupi,43,83,27,14,167,19.6
Porto Nacional,28,58,34,3,123,14.4
Miracema,6,10,15,4,35,4.1
Reitoria,9,0,3,14,26,3.0
Araguaína,1,0,0,0,1,0.1
TOTAL,182,464,137,71,854,100.0


### Perfis dos Respondentes

#### Perfil dos Discentes

In [113]:
discentes = df[df['segmento'] == 'Discente']
total_discentes = len(discentes)

if total_discentes > 0:
  nivel_cols = [col for col in df.columns if 'NIVEL' in col.upper() and 'CURSO' in col.upper()]
  if nivel_cols:
    nivel_col = nivel_cols[0]
    nivel_counts = discentes[nivel_col].value_counts()
    print(f"\nDistribuição por nível de curso:")
    for nivel, count in nivel_counts.items():
      pct = (count / total_discentes) * 100
      print(f"  {nivel:<40} {count:>4} ({pct:>5.1f}%)")

  tipo_cols = [col for col in df.columns if 'TIPO' in col.upper() and 'CURSO' in col.upper()]
  if tipo_cols:
    tipo_col = tipo_cols[0]
    tipo_counts = discentes[tipo_col].value_counts()
    print(f"\nDistribuição por tipo de curso:")
    for tipo, count in tipo_counts.items():
      pct = (count / total_discentes) * 100
      print(f"  {tipo:<40} {count:>4} ({pct:>5.1f}%)")

  curso_cols = [col for col in df.columns if 'NOME' in col.upper() and 'CURSO' in col.upper()]
  if curso_cols:
    curso_col = curso_cols[0]
    curso_counts = discentes[curso_col].value_counts()
    print(f"\nTop 10 cursos com mais respondentes:")
    for i, (curso, count) in enumerate(curso_counts.head(10).items(), 1):
      pct = (count / total_discentes) * 100
      print(f"  {i:>2}. {str(curso):<55} {count:>3} ({pct:>4.1f}%)")
else:
  print("⚠️ Nenhum discente encontrado")


Distribuição por nível de curso:
  Nível de Graduação                        415 ( 89.4%)
  Nível de Pós-Graduação                     23 (  5.0%)
  Especial de Pós Graduação                  18 (  3.9%)
  Nível de Especialização                     7 (  1.5%)
  Nível de Graduação/Pronera                  1 (  0.2%)

Distribuição por tipo de curso:
  NORMAL                                    456 ( 98.3%)
  EAD                                         8 (  1.7%)

Top 10 cursos com mais respondentes:
   1. Curso de Pedagogia                                       77 (16.6%)
   2. Curso de Agronomia                                       51 (11.0%)
   3. Curso de Educação do Campo: Códigos e Linguagem - Artes Visuais e Música  47 (10.1%)
   4. Curso de Ciência da Computação                           40 ( 8.6%)
   5. Curso de Arquitetura e Urbanismo                         24 ( 5.2%)
   6. Curso de Engenharia Florestal                            16 ( 3.4%)
   7. Curso de Ciências Contábeis  

#### Perfil dos Docentes

In [114]:
docentes = df[df['segmento'] == 'Docente']
total_docentes = len(docentes)

if total_docentes > 0:
  titulacao_cols = [col for col in df.columns if 'TITULACAO' in col.upper() and 'PROFESSOR' in col.upper()]
  if titulacao_cols:
    titulacao_col = titulacao_cols[0]
    titulacao_counts = docentes[titulacao_col].value_counts()
    print(f"\nDistribuição por titulação:")
    for tit, count in titulacao_counts.items():
      pct = (count / total_docentes) * 100
      print(f"  {tit:<40} {count:>4} ({pct:>5.1f}%)")
else:
  print("⚠️ Nenhum docente encontrado")


Distribuição por titulação:
  Doutorado                                 125 ( 91.2%)
  Mestrado                                   12 (  8.8%)


#### Perfil dos Técnicos

In [115]:
tecnicos = df[df['segmento'] == 'Técnico']
total_tecnicos = len(tecnicos)

if total_tecnicos > 0:
  titulacao_cols = [col for col in df.columns if 'TITULACAO' in col.upper() and 'TECNICO' in col.upper()]
  if titulacao_cols:
    titulacao_col = titulacao_cols[0]
    titulacao_counts = tecnicos[titulacao_col].value_counts()
    print(f"\nDistribuição por titulação:")
    for tit, count in titulacao_counts.items():
      pct = (count / total_tecnicos) * 100
      print(f"  {tit:<45} {count:>4} ({pct:>5.1f}%)")
else:
  print("⚠️ Nenhum técnico encontrado")


Distribuição por titulação:
  Especialização                                  29 ( 40.8%)
  Mestrado                                        27 ( 38.0%)
  Doutorado                                        5 (  7.0%)
  Graduação                                        4 (  5.6%)
  Superior Incompleto                              3 (  4.2%)
  2 Grau Completo ou Técnico                       2 (  2.8%)
  Ensino Médio Profissionalizante / com Curso Técnico Completo    1 (  1.4%)


#### Perfil dos Egressos

In [116]:
egressos = df[df['segmento'] == 'Egresso']
total_egressos = len(egressos)

if total_egressos > 0:
    colunas_curso = {
        'EGRESSO_NIVEL_CURSO': 'Nível do Curso',
        'EGRESSO_NOME_CURSO_DIPLOMA': 'Nome do Curso/Diploma'
    }

    for col, descricao in colunas_curso.items():
        if col in egressos.columns and egressos[col].notna().sum() > 0:
            curso_counts = (
                egressos[col]
                .value_counts(dropna=True)
                .sort_index()
            )

            print(f"\n📊 Distribuição de egressos por {descricao} ({col}):\n")

            for curso, count in curso_counts.items():
                pct = (count / total_egressos) * 100
                print(f"  {str(curso)[:60]:<60} {count:>4} ({pct:>5.1f}%)")

            # Criação da tabela formatada
            tabela = (
                curso_counts
                .rename_axis(descricao)
                .reset_index(name='Quantidade')
            )
            tabela['%'] = (tabela['Quantidade'] / total_egressos * 100).round(1)
            tabela = tabela.sort_values('Quantidade', ascending=False)



📊 Distribuição de egressos por Nível do Curso (EGRESSO_NIVEL_CURSO):

  Especial de Graduação                                           3 (  1.6%)
  Especial de Pós Graduação                                       3 (  1.6%)
  Nível de Especialização                                         4 (  2.2%)
  Nível de Graduação                                            138 ( 75.8%)
  Nível de Graduação/PARFOR                                       1 (  0.5%)
  Nível de Pós-Graduação                                         33 ( 18.1%)

📊 Distribuição de egressos por Nome do Curso/Diploma (EGRESSO_NOME_CURSO_DIPLOMA):

  Administração                                                   4 (  2.2%)
  Análise de Dados                                                2 (  1.1%)
  Análise de Dados de Controle                                    1 (  0.5%)
  Computação                                                      1 (  0.5%)
  Curso Especial da Graduação                                     3 (  1.6

#### Estrutura das Questões por Eixo

In [117]:
print("\n📚 8. ESTRUTURA DAS QUESTÕES POR EIXO SINAES")
print("-"*80)

catalogo_eixos = {
    "Eixo 1 - Planejamento e Avaliação": {
        'questoes': ['q1', 'q1.1', 'q2', 'q2.1', 'q3', 'q3.1', 'q4', 'q4.1', 'q5', 'q6'],
        'range': 'q1–q6'
    },
    "Eixo 2 - Desenvolvimento Institucional": {
        'questoes': [f'q{i}' for i in range(7, 13)],
        'range': 'q7–q12'
    },
    "Eixo 3 - Políticas Acadêmicas": {
        'questoes': [f'q{i}' for i in range(13, 32)] + ['q29.1'],
        'range': 'q13–q31'
    },
    "Eixo 4 - Políticas de Gestão": {
        'questoes': [f'q{i}' for i in range(32, 43)],
        'range': 'q32–q42'
    },
    "Eixo 5 - Infraestrutura": {
        'questoes': [f'q{i}' for i in range(43, 60)] + ['q70'],
        'range': 'q43–q59'
    }
}

print(f"\n{'Eixo':<45} {'Questões':<10} {'Range':<12}")
print("-"*80)

total_questoes = 0
for eixo, dados in catalogo_eixos.items():
    n_questoes = len([q for q in dados['questoes'] if q in df.columns or f"{q}_num" in df.columns])
    total_questoes += n_questoes
    print(f"{eixo:<45} {n_questoes:<10} {dados['range']:<12}")

print("-"*80)
print(f"{'TOTAL':<45} {total_questoes:<10}")


📚 8. ESTRUTURA DAS QUESTÕES POR EIXO SINAES
--------------------------------------------------------------------------------

Eixo                                          Questões   Range       
--------------------------------------------------------------------------------
Eixo 1 - Planejamento e Avaliação             10         q1–q6       
Eixo 2 - Desenvolvimento Institucional        6          q7–q12      
Eixo 3 - Políticas Acadêmicas                 20         q13–q31     
Eixo 4 - Políticas de Gestão                  11         q32–q42     
Eixo 5 - Infraestrutura                       18         q43–q59     
--------------------------------------------------------------------------------
TOTAL                                         65        


## 2.5. Converter para Numérico

In [118]:
# vamos tirar as questões binarias para nao misturar escalas
colunas_questoes_likert = []

for col in colunas_questoes:
    if col in df.columns:
        amostra = df[col].dropna().astype(str).str.strip().str.lower().head(20)
        tem_binaria = any(val in ['s', 'n', 'sim', 'não', 'nao'] for val in amostra)

        if not tem_binaria:
            colunas_questoes_likert.append(col)

for col in colunas_questoes_likert:
    df[f"{col}_num"] = df[col].apply(to_likert_numeric)

colunas_num = [f"{col}_num" for col in colunas_questoes_likert if f"{col}_num" in df.columns]

print(f"{len(colunas_num)} questões Likert convertidas")
print(f"{len(colunas_questoes) - len(colunas_questoes_likert)} questões S/N ignoradas")


57 questões Likert convertidas
8 questões S/N ignoradas



# PARTE 3: ANÁLISE EXPLORATÓRIA

#### Distribuiçaõ agregada das avaliacoes por faixas da escala

In [119]:
import numpy as np
import pandas as pd

tabela_eixos = []

for eixo_nome, dados in catalogo_eixos.items():
    valores = []
    for q in dados['questoes']:
        col = f"{q}_num" if f"{q}_num" in df.columns else q
        if col in df.columns:
            vals = df[col].dropna().tolist()
            valores.extend(vals)

    if not valores:
        continue

    valores = pd.to_numeric(pd.Series(valores), errors='coerce').dropna()

    if len(valores) == 0:
        continue

    n = len(valores)
    n_1_2 = ((valores >= 1) & (valores <= 2)).sum()
    n_3_4 = ((valores >= 3) & (valores <= 4)).sum()
    n_5_6 = ((valores >= 5) & (valores <= 6)).sum()

    pct_1_2 = (n_1_2 / n) * 100
    pct_3_4 = (n_3_4 / n) * 100
    pct_5_6 = (n_5_6 / n) * 100
    gap = pct_5_6 - pct_1_2

    tabela_eixos.append({
        'Eixo': eixo_nome.split(" - ")[1],
        '1–2 (%)': round(pct_1_2, 1),
        '3–4 (%)': round(pct_3_4, 1),
        '5–6 (%)': round(pct_5_6, 1),
        'Gap (pp)': f"{gap:+.1f}"
    })

df_eixos_fmt = pd.DataFrame(tabela_eixos)
display(df_eixos_fmt)


,Eixo,1–2 (%),3–4 (%),5–6 (%),Gap (pp)
0,Planejamento e Avaliação,9.5,38.5,52.0,+42.5
1,Desenvolvimento Institucional,12.7,35.2,52.2,+39.5
2,Políticas Acadêmicas,19.0,37.2,43.8,+24.7
3,Políticas de Gestão,25.2,37.1,37.7,+12.5
4,Infraestrutura,27.5,35.3,37.2,+9.7


#### Avaliação por Eixo

In [120]:
import numpy as np
import pandas as pd

resultados_eixos = []

for eixo_nome, dados in catalogo_eixos.items():
    valores = []
    for q in dados['questoes']:
        col = f"{q}_num" if f"{q}_num" in df.columns else q
        if col in df.columns:
            vals = df[col].dropna().tolist()
            valores.extend(vals)

    if len(valores) == 0:
        continue

    valores = pd.Series(valores)
    valores = pd.to_numeric(valores, errors='coerce').dropna()

    if len(valores) == 0:
        continue

    n = len(valores)
    media = valores.mean()

    # cálculo das faixas
    n_1_2 = ((valores >= 1) & (valores <= 2)).sum()
    n_3_4 = ((valores >= 3) & (valores <= 4)).sum()
    n_5_6 = ((valores >= 5) & (valores <= 6)).sum()

    pct_1_2 = (n_1_2 / n) * 100
    pct_3_4 = (n_3_4 / n) * 100
    pct_5_6 = (n_5_6 / n) * 100

    resultados_eixos.append({
        'Eixo': eixo_nome,
        'N': n,
        'Média': round(media, 2),
        '% Notas 1-2': round(pct_1_2, 1),
        '% Notas 3-4': round(pct_3_4, 1),
        '% Notas 5-6': round(pct_5_6, 1)
    })

df_eixos = pd.DataFrame(resultados_eixos)
display(df_eixos)


,Eixo,N,Média,% Notas 1-2,% Notas 3-4,% Notas 5-6
0,Eixo 1 - Planejamento e Avaliação,1041,4.39,9.5,38.5,52.0
1,Eixo 2 - Desenvolvimento Institucional,4780,4.34,12.7,35.2,52.2
2,Eixo 3 - Políticas Acadêmicas,12452,4.02,19.0,37.2,43.8
3,Eixo 4 - Políticas de Gestão,7001,3.74,25.2,37.1,37.7
4,Eixo 5 - Infraestrutura,11962,3.68,27.5,35.3,37.2


## 3.1. Estatísticas Descritivas

In [121]:
resultados = []
for col_num in colunas_num:
    stats_desc = estatisticas_descritivas(df[col_num])
    if stats_desc:
        stats_desc['questao'] = col_num.replace('_num', '')
        resultados.append(stats_desc)

df_stats = pd.DataFrame(resultados)
df_stats = df_stats[['questao', 'n', 'media', 'mediana', 'desvio_padrao']]

print("Primeiras 15 questões:")
display(df_stats.head(15))

Primeiras 15 questões:


,questao,n,media,mediana,desvio_padrao
0,q1.1,331,4.054381,4.0,1.376228
1,q2.1,262,4.366412,4.0,1.305492
2,q3.1,261,4.659004,5.0,1.177761
3,q4.1,187,4.647059,5.0,1.241543
4,q7,825,4.532121,5.0,1.309691
5,q8,828,4.115942,4.0,1.512091
6,q9,828,4.091787,4.0,1.440905
7,q10,751,4.227696,4.0,1.457882
8,q11,770,4.624675,5.0,1.401528
9,q12,778,4.452442,5.0,1.502782


## 3.2. Ranking de Questões

In [122]:
df_ranking = df_stats.sort_values('media', ascending=False)

print("\nmaior satisfacao:")
display(df_ranking.head(10))

print("\n menor satisfacao:")
display(df_ranking.tail(10))


maior satisfacao:


,questao,n,media,mediana,desvio_padrao
2,q3.1,261,4.659004,5.0,1.177761
3,q4.1,187,4.647059,5.0,1.241543
8,q11,770,4.624675,5.0,1.401528
4,q7,825,4.532121,5.0,1.309691
23,q26,805,4.524224,5.0,1.432901
30,q34,731,4.476060,5.0,1.604075
9,q12,778,4.452442,5.0,1.502782
52,q56,838,4.446301,5.0,1.547760
17,q20,740,4.387838,5.0,1.447171
16,q19,758,4.375989,5.0,1.364701



 menor satisfacao:


,questao,n,media,mediana,desvio_padrao
33,q37,630,3.500000,4.0,1.637935
32,q36,679,3.474227,4.0,1.744481
51,q55,774,3.453488,4.0,1.755356
35,q39,651,3.453149,4.0,1.597749
44,q48,299,3.444816,4.0,1.680763
40,q44,713,3.434783,3.0,1.699809
37,q41,551,3.335753,3.0,1.653033
55,q59,674,3.249258,3.0,1.731675
42,q46,842,3.211401,3.0,1.640876
49,q53,715,2.827972,2.0,1.815680


## 3.3. Análise por Eixo SINAES

In [123]:
medias_eixos_list = []

for eixo_nome, eixo_info in EIXOS_SINAES.items():
    questoes_eixo_num = [f"{q}_num" for q in eixo_info['questoes'] if f"{q}_num" in colunas_num]

    if questoes_eixo_num:
        valores = df[questoes_eixo_num].values.flatten()
        valores_validos = valores[~np.isnan(valores)]

        if len(valores_validos) > 0:
            medias_eixos_list.append({
                'eixo': eixo_nome,
                'nome': eixo_info['nome'],
                'media': np.mean(valores_validos),
                'desvio_padrao': np.std(valores_validos),
                'n_questoes': len(questoes_eixo_num),
                'cor': eixo_info['cor']
            })

medias_eixos = pd.DataFrame(medias_eixos_list).sort_values('media', ascending=False)

display(medias_eixos[['eixo', 'nome', 'media', 'desvio_padrao', 'n_questoes']])

,eixo,nome,media,desvio_padrao,n_questoes
0,Eixo 1,Planejamento e Avaliação Institucional,4.390970,1.309786,4
1,Eixo 2,Desenvolvimento Institucional,4.337866,1.452551,6
2,Eixo 3,Políticas Acadêmicas,4.015660,1.570997,18
3,Eixo 4,Políticas de Gestão,3.742180,1.656454,11
4,Eixo 5,Infraestrutura Física,3.678816,1.723196,17


## 3.4. Análise por Segmento

In [124]:
resultados_seg = []

for seg in df[coluna_segmento].unique():
    df_seg = df[df[coluna_segmento] == seg]

    valores_seg = df_seg[colunas_num].values.flatten()
    valores_seg_validos = valores_seg[~np.isnan(valores_seg)]

    if len(valores_seg_validos) > 0:
        resultados_seg.append({
            'segmento': seg,
            'n_respondentes': len(df_seg),
            'satisfacao_media': np.mean(valores_seg_validos),
            'satisfacao_mediana': np.median(valores_seg_validos),
            'desvio_padrao': np.std(valores_seg_validos)
        })

df_seg_stats = pd.DataFrame(resultados_seg).sort_values('satisfacao_media', ascending=False)

print("Satisfação Geral por Segmento:")
display(df_seg_stats)

Satisfação Geral por Segmento:


,segmento,n_respondentes,satisfacao_media,satisfacao_mediana,desvio_padrao
3,Técnico,71,4.006017,4.0,1.455612
1,Discente,464,3.982101,4.0,1.683426
2,Egresso,182,3.953339,4.0,1.628868
0,Docente,137,3.593175,4.0,1.516178



# PARTE 4: ANÁLISE INFERENCIAL

## 4.1. Matriz de Correlação

In [125]:
df[colunas_num] = df[colunas_num].apply(pd.to_numeric, errors='coerce')

matriz_corr = df[colunas_num].corr(method='spearman', min_periods=3)

mask_validas = matriz_corr.notna().any(axis=1)
matriz_corr_filtrada = matriz_corr.loc[mask_validas, mask_validas]

print(f"✅ {matriz_corr_filtrada.shape[0]} colunas com correlação válida.")
display(matriz_corr_filtrada)

print(df[colunas_num].head(5))


✅ 56 colunas com correlação válida.


,q1.1_num,q2.1_num,q3.1_num,q4.1_num,q7_num,q8_num,q9_num,q10_num,q11_num,q12_num,...,q50_num,q51_num,q52_num,q53_num,q54_num,q55_num,q56_num,q57_num,q58_num,q59_num
q1.1_num,1.000000,0.812606,0.426343,0.573243,0.634303,0.630698,0.667779,0.611110,0.553236,0.512313,...,0.638544,0.525833,0.509641,0.480065,0.469619,0.536139,0.411693,0.435933,0.588007,0.577883
q2.1_num,0.812606,1.000000,0.625642,0.703869,0.667541,0.642395,0.677798,0.627275,0.559449,0.488749,...,0.622136,0.611295,0.504260,0.512483,0.480876,0.635370,0.460657,0.459165,0.626411,0.597434
q3.1_num,0.426343,0.625642,1.000000,0.842103,0.509316,0.419848,0.433329,0.484189,0.346506,0.309001,...,0.336900,0.368217,0.302440,0.274234,0.199895,0.371294,0.229478,0.180376,0.306821,0.293554
q4.1_num,0.573243,0.703869,0.842103,1.000000,0.531389,0.503295,0.443894,0.505843,0.438026,0.454933,...,0.511921,0.532940,0.376530,0.324665,0.199147,0.437608,0.218756,0.229231,0.466763,0.298481
q7_num,0.634303,0.667541,0.509316,0.531389,1.000000,0.708480,0.722349,0.675948,0.577676,0.556955,...,0.457726,0.454354,0.414498,0.383833,0.301357,0.477669,0.359704,0.340396,0.518583,0.419000
q8_num,0.630698,0.642395,0.419848,0.503295,0.708480,1.000000,0.765181,0.678744,0.526746,0.562997,...,0.509283,0.478502,0.423778,0.366681,0.238965,0.506558,0.368293,0.284451,0.490549,0.422334
q9_num,0.667779,0.677798,0.433329,0.443894,0.722349,0.765181,1.000000,0.728630,0.569720,0.547686,...,0.513558,0.487230,0.475849,0.418050,0.285141,0.529557,0.385745,0.301259,0.528563,0.473313
q10_num,0.611110,0.627275,0.484189,0.505843,0.675948,0.678744,0.728630,1.000000,0.602097,0.560979,...,0.471342,0.455787,0.425049,0.405308,0.340857,0.503337,0.351649,0.332415,0.507680,0.430649
q11_num,0.553236,0.559449,0.346506,0.438026,0.577676,0.526746,0.569720,0.602097,1.000000,0.685895,...,0.421026,0.448212,0.361591,0.394081,0.285304,0.449344,0.345203,0.299263,0.549782,0.433338
q12_num,0.512313,0.488749,0.309001,0.454933,0.556955,0.562997,0.547686,0.560979,0.685895,1.000000,...,0.390369,0.424909,0.343320,0.364469,0.314280,0.451060,0.351974,0.288351,0.492002,0.446133


   q1.1_num  q2.1_num  q3.1_num  q4.1_num  q7_num  q8_num  q9_num  q10_num  \
0       3.0       3.0       6.0       6.0     4.0     3.0     2.0      4.0   
1       NaN       NaN       NaN       NaN     4.0     3.0     1.0      4.0   
2       NaN       NaN       NaN       NaN     4.0     NaN     NaN      5.0   
3       NaN       NaN       NaN       NaN     3.0     3.0     2.0      NaN   
4       4.0       3.0       1.0       1.0     5.0     5.0     4.0      4.0   

   q11_num  q12_num  ...  q51_num  q52_num  q53_num  q54_num  q55_num  \
0      3.0      6.0  ...      3.0      4.0      1.0      1.0      1.0   
1      1.0      NaN  ...      3.0      4.0      2.0      2.0      NaN   
2      5.0      6.0  ...      5.0      3.0      NaN      4.0      4.0   
3      5.0      2.0  ...      6.0      6.0      1.0      1.0      1.0   
4      5.0      5.0  ...      1.0      3.0      1.0      NaN      NaN   

   q56_num  q57_num  q58_num  q59_num  q70_num  
0      4.0      1.0      5.0      1.0      

## 4.2. Identificar Correlações Fortes

In [126]:
correlacoes_fortes = []

for i in range(len(matriz_corr.columns)):
    for j in range(i+1, len(matriz_corr.columns)):
        corr = matriz_corr.iloc[i, j]
        if abs(corr) > 0.7:
            correlacoes_fortes.append({
                'questao_1': matriz_corr.columns[i],
                'questao_2': matriz_corr.columns[j],
                'correlacao': corr,
                'interpretacao': interpretar_correlacao(corr)
            })

if len(correlacoes_fortes) > 0:
    df_corr = pd.DataFrame(correlacoes_fortes).sort_values('correlacao', key=abs, ascending=False)
    print(f"\n🔍 {len(df_corr)} correlações fortes encontradas (|ρ| > 0.7):\n")
    display(df_corr.head(10))
else:
    print("\n⚠️  Nenhuma correlação forte (>0.7) encontrada no subset analisado.")
    print("   Isso pode ser normal - tente aumentar o número de questões ou reduzir o threshold. Coisas")


🔍 60 correlações fortes encontradas (|ρ| > 0.7):



,questao_1,questao_2,correlacao,interpretacao
44,q37_num,q39_num,0.847458,Correlação positiva forte
7,q3.1_num,q4.1_num,0.842103,Correlação positiva forte
56,q44_num,q45_num,0.832327,Correlação positiva forte
21,q21_num,q22_num,0.830255,Correlação positiva forte
28,q32_num,q37_num,0.812943,Correlação positiva forte
0,q1.1_num,q2.1_num,0.812606,Correlação positiva forte
26,q32_num,q35_num,0.808266,Correlação positiva forte
39,q36_num,q37_num,0.804539,Correlação positiva forte
35,q35_num,q37_num,0.798487,Correlação positiva forte
43,q37_num,q38_num,0.791781,Correlação positiva forte


## 4.3. Teste Estatístico entre Segmentos

Nota: ver se iremos usar isso mesmo ou meh

In [127]:
print("\nKRUSKAL-WALLIS - maybe ...\n")

questao_teste = colunas_num[0]
print(f"Questão testada: {questao_teste}\n")

grupos = []
segmentos = df[coluna_segmento].unique()

for seg in segmentos:
    vals = df[df[coluna_segmento] == seg][questao_teste].dropna()
    if len(vals) > 0:
        grupos.append(vals)

if len(grupos) >= 2:
    h_stat, p_valor = stats.kruskal(*grupos)

    print(f"Resultados:")
    print(f"  H-estatística: {h_stat:.4f}")
    print(f"  p-valor: {p_valor:.4e}")
    print(f"  Significativo (α=0.05)? {p_valor < 0.05}")

    if p_valor < 0.05:
        print("\ndiferencas a considerar")


KRUSKAL-WALLIS - maybe ...

Questão testada: q1.1_num

Resultados:
  H-estatística: 12.5883
  p-valor: 5.6170e-03
  Significativo (α=0.05)? True

diferencas a considerar



# pARTE 5: VISUALIZAÇÕES

## 5.1. Gráfico de Barras

In [128]:
fig_eixos = go.Figure()

fig_eixos.add_bar(
    x=medias_eixos['eixo'],
    y=medias_eixos['media'],
    marker=dict(
        color=medias_eixos['cor'],
        line=dict(color='black', width=1.5)
    ),
    text=[f"{m:.2f}" for m in medias_eixos['media']],
    textposition='outside',
    textfont=dict(size=14, color='black')
)

fig_eixos.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Eixo SINAES",
        titlefont=dict(size=18),
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        title="Média (escala 1-6)",
        range=[0, 6.5],
        titlefont=dict(size=18),
        tickfont=dict(size=14)
    ),
    margin=dict(l=80, r=40, t=70, b=120),
    width=900,
    height=550,
    showlegend=False
)

fig_eixos.add_hline(
    y=4.0,
    line_dash="dash",
    line_color="red",
    opacity=0.5,
    annotation_text="Referência (4.0)",
    annotation_position="right"
)

fig_eixos.show()

##### Graficos q1 e q3

In [129]:
base = df[["segmento", "q1", "q3"]].copy()

base.rename(columns={'segmento': 'grupo'}, inplace=True)

cores_dict = {
    "Sim": "#06d6a0",
    "Não": "#ff6b6b",
    "Não soube opinar": "#adb5bd"
}

def criar_grafico(base_dados, coluna_questao, titulo_grafico=""):
    base_dados = base_dados.copy()
    base_dados.loc[:, "resposta"] = base_dados[coluna_questao].map({
        "s": "Sim",
        "n": "Não",
        "nso": "Não soube opinar"
    })

    contagem = base_dados.groupby(["grupo", "resposta"]).size().reset_index(name="n")
    total = base_dados.groupby("grupo").size().reset_index(name="total")
    resultado = contagem.merge(total)
    resultado["perc"] = (resultado["n"] / resultado["total"]) * 100

    ordem_segmentos = [seg for seg in ["Discente", "Docente", "Técnico", "Egresso"] if seg in base_dados["grupo"].unique()]

    fig = go.Figure()

    for resp in ["Sim", "Não", "Não soube opinar"]:
        dados_resp = resultado[resultado["resposta"] == resp]
        fig.add_bar(
            x=dados_resp["grupo"],
            y=dados_resp["perc"],
            name=resp,
            marker_color=cores_dict[resp],
            hovertemplate="%{x}<br>" + resp + ": %{y:.1f}%<extra></extra>"
        )

    fig.update_layout(
        barmode="stack",
        template="plotly_white",
        #title=titulo_grafico,
        yaxis=dict(
            title="Percentual de respostas",
            ticksuffix="%",
            titlefont=dict(size=18),
            tickfont=dict(size=16),
            range=[0, 100]
        ),
        xaxis=dict(
            title="Segmento",
            titlefont=dict(size=18),
            tickfont=dict(size=16),
            categoryorder="array",
            categoryarray=ordem_segmentos
        ),
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.02,
            font=dict(size=15),
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="gray",
            borderwidth=1.5
        ),
        margin=dict(l=80, r=150, t=40, b=80),
        height=500,
        width=900
    )

    return fig

fig_pdi = criar_grafico(base, "q1", titulo_grafico="Questão 1: PDI é claro para a comunidade?")
html_output_path_pdi = OUTPUTS_DIR / "html" / "grafico_pdi.html"
html_output_path_pdi.parent.mkdir(parents=True, exist_ok=True)
fig_pdi.write_html(str(html_output_path_pdi),
                   config={'toImageButtonOptions': {'format': 'svg', 'width': 900, 'height': 500}})

fig_cpa = criar_grafico(base, "q3", titulo_grafico="Questão 3: CPA é conhecida pela comunidade?")
html_output_path_cpa = OUTPUTS_DIR / "html" / "grafico_cpa.html"
html_output_path_cpa.parent.mkdir(parents=True, exist_ok=True)
fig_cpa.write_html(str(html_output_path_cpa),
                   config={'toImageButtonOptions': {'format': 'svg', 'width': 900, 'height': 500}})

#### Media das respostas com intervalo de confianca

Adotamos aqui 95% por convenção estatística, mas podemos baixar para 90% se quisermos ser mais pragmaticos. É intervalo de confiança pq o recorte dos dados é uma pequena população da comunidade academica

In [130]:
registros = []
todos_valores = []

for eixo_key, eixo_info in EIXOS_SINAES.items():
    questoes_eixo_num = [f"{q}_num" for q in eixo_info['questoes']
                         if f"{q}_num" in df.columns]

    if not questoes_eixo_num:
        continue

    vals = df[questoes_eixo_num].values.flatten()
    vals = vals[~np.isnan(vals)]

    if len(vals) == 0:
        continue

    # Adiciona à lista geral
    todos_valores.extend(vals)

    media = np.mean(vals)

    sem = stats.sem(vals)
    ic = sem * stats.t.ppf((1 + 0.95) / 2., len(vals)-1)

    registros.append({
        "Eixo": eixo_name_to_display[eixo_info['nome']],
        "Média": media,
        "IC_lower": media - ic,
        "IC_upper": media + ic,
        "N": len(vals)
    })

res = pd.DataFrame(registros)
res["Eixo"] = pd.Categorical(res["Eixo"], categories=ordem_eixos_display, ordered=True)
res = res.sort_values("Eixo")

# Calcula média geral de TODAS as respostas
media_geral = np.mean(todos_valores)

# Cores diferentes por eixo (estilo do exemplo)
cores_eixos = ['#06D6A0', '#5BC0EB', '#FFC857', '#EF476F', '#118AB2']

# Cria gráfico de barras com intervalo de confiança
fig = go.Figure()

fig.add_trace(go.Bar(
    x=res["Eixo"],
    y=res["Média"],
    marker=dict(
        color=cores_eixos[:len(res)],
        line=dict(color='rgba(0,0,0,0.3)', width=1)
    ),
    error_y=dict(
        type='data',
        symmetric=False,
        array=res["IC_upper"] - res["Média"],
        arrayminus=res["Média"] - res["IC_lower"],
        color='rgba(0,0,0,0.8)',
        thickness=2,
        width=6
    ),
    text=res["Média"].round(2),
    textposition='outside',
    textfont=dict(size=14, color='black'),
    hovertemplate=(
        "<b>%{x}</b><br>" +
        "Média: %{y:.2f}<br>" +
        "IC 95%%: [%{customdata[0]:.2f} - %{customdata[1]:.2f}]<br>" +
        "N respondentes: %{customdata[2]}" +
        "<extra></extra>"
    ),
    customdata=np.column_stack((res["IC_lower"], res["IC_upper"], res["N"]))
))

fig.add_hline(
    y=media_geral,
    line_dash="dash",
    line_color="rgba(100,100,100,0.6)",
    line_width=2,
    annotation_text=f"Ponto médio: {media_geral:.2f}",
    annotation_position="right",
    annotation_font_size=12,
    annotation_font_color="rgba(100,100,100,0.8)"
)

fig.update_layout(
    template="plotly_white",
    yaxis=dict(
        title="Média de avaliação (escala 1–6)",
        range=[0, 5.5],
        tickmode="linear",
        tick0=0,
        dtick=1,
        titlefont=dict(size=16),
        tickfont=dict(size=14),
        gridcolor='rgba(200,200,200,0.3)'
    ),
    xaxis=dict(
        title="Eixos SINAES",
        titlefont=dict(size=16),
        tickfont=dict(size=12),
        tickangle=-15
    ),
    margin=dict(l=80, r=80, t=60, b=120),
    height=550,
    width=950,
    showlegend=False,
    plot_bgcolor='white'
)

fig.show()

html_output_path = OUTPUTS_DIR / "html" / "grafico_barras_eixos_ic.html"
html_output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(html_output_path),
               config={'toImageButtonOptions': {'format': 'svg', 'width': 950, 'height': 550}})

for _, row in res.iterrows():
    print(f"   {row['Eixo']}: {row['Média']:.2f} (IC: [{row['IC_lower']:.2f} - {row['IC_upper']:.2f}], N={row['N']})")
print(f"\nmedia: {media_geral:.2f}")

   Planejamento e
Avaliação: 4.39 (IC: [4.31 - 4.47], N=1041)
   Desenvolvimento
Institucional: 4.34 (IC: [4.30 - 4.38], N=4780)
   Políticas
Acadêmicas: 4.02 (IC: [3.99 - 4.04], N=12452)
   Políticas de
Gestão: 3.74 (IC: [3.70 - 3.78], N=7001)
   Infraestrutura: 3.68 (IC: [3.65 - 3.71], N=11962)

media: 3.91


## 5.2. Boxplot - Comparação por Segmento

In [131]:
questao_viz = colunas_num[min(10, len(colunas_num)-1)]

print(f"\nBoxplot para: {questao_viz}\n")

import numpy as np

fig_box = go.Figure()

segmentos = sorted(df[coluna_segmento].unique())

# Cor única profissional
cor_unica = '#2c3e50'

for i, seg in enumerate(segmentos):
    vals = df[df[coluna_segmento] == seg][questao_viz].dropna()

    if len(vals) > 0:
        # Criar jitter manual
        np.random.seed(42)
        jitter_strength = 0.12
        x_jittered = np.random.normal(i, jitter_strength, size=len(vals))

        # Pontos individuais
        fig_box.add_trace(go.Scatter(
            x=x_jittered,
            y=vals,
            mode='markers',
            name=seg,
            marker=dict(
                color=cor_unica,
                size=5,
                opacity=0.35,
                line=dict(width=0.3, color='white')
            ),
            showlegend=False,
            hovertemplate=f'<b>{seg}</b><br>Nota: %{{y}}<extra></extra>'
        ))

        # Boxplot minimalista
        fig_box.add_trace(go.Box(
            y=vals,
            x=[i] * len(vals),
            name=seg,
            boxmean=False,
            marker=dict(
                color=cor_unica,
                size=0
            ),
            line=dict(width=1.5, color=cor_unica),
            fillcolor='rgba(0,0,0,0)',
            width=0.5,
            showlegend=False,
            hovertemplate=f'<b>{seg}</b><br>Mediana: %{{median}}<br>Q1: %{{q1}}<br>Q3: %{{q3}}<extra></extra>'
        ))

fig_box.update_layout(
    template="plotly_white",
    yaxis=dict(
        title="Nota",
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5, 6],
        range=[0.5, 6.5],
        titlefont=dict(size=14, color="#333"),
        tickfont=dict(size=12, color="#555"),
        gridcolor='#e8e8e8',
        gridwidth=0.8,
        showline=True,
        linewidth=1,
        linecolor='#ccc',
        zeroline=False
    ),
    xaxis=dict(
        title="Segmento",
        titlefont=dict(size=14, color="#333"),
        tickfont=dict(size=12, color="#555"),
        tickmode='array',
        tickvals=list(range(len(segmentos))),
        ticktext=segmentos,
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor='#ccc'
    ),
    plot_bgcolor='#fafafa',
    paper_bgcolor='white',
    margin=dict(l=70, r=40, t=40, b=70),
    width=800,
    height=500,
    showlegend=False,
    hovermode='closest'
)

fig_box.show()


Boxplot para: q13_num



Grafico de Violino - Não ficou bom -> faz mais sentido em variáveis contínuas

In [132]:
fig_violin = go.Figure()

for seg in df[coluna_segmento].unique():
    vals = df[df[coluna_segmento] == seg][questao_viz].dropna()

    if len(vals) > 0:
        fig_violin.add_trace(go.Violin(
            y=vals,
            x=[seg] * len(vals),
            name=seg,
            box_visible=True,
            meanline_visible=True,
            line_color='darkgrey',
            fillcolor='rgba(200, 200, 200, 0.5)',
            marker_color='grey',
            points=False,
            hovertemplate=f"<b>Segmento:</b> {seg}<br>" +
                          "<b>Mediana:</b> %{{y}} (e.g.)<extra></extra>"
        ))

fig_violin.update_layout(
    template="plotly_white",
    yaxis=dict(
        title="Nota (1–6)",
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5, 6],
        range=[0.5, 6.5],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    xaxis=dict(
        title="Segmento",
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    margin=dict(l=80, r=40, t=70, b=70),
    width=800,
    height=500,
    showlegend=False
)

fig_violin.show()

## 5.3. Heatmap - Matriz de Correlação

#### Q17 - Acesso a internet x Qualidade dos Ambientes Virtuais

In [133]:
base = df[["q46_num", "q17_num"]].dropna().copy()

# calc
r_pearson, p_pearson = stats.pearsonr(base["q46_num"], base["q17_num"])
rho_spearman, p_spearman = stats.spearmanr(base["q46_num"], base["q17_num"])

print("="*60)
print("CORRELACAO")
print("="*60)
print(f"   Variável X: Acesso a internet (q46)")
print(f"   Variável Y: AVA/Moodle (q17)")
print(f"   N = {len(base)} respostas válidas")
print()
print(f"   ->> Correlação de Pearson:")
print(f"     r = {r_pearson:.4f}")
print(f"     p-valor = {p_pearson:.4e}")
print(f"     {'✅ Significativa' if p_pearson < 0.05 else '⚠️ Não significativa'} (α = 0.05)")
print()
print(f"   ->> Correlação de Spearman:")
print(f"     ρ = {rho_spearman:.4f}")
print(f"     p-valor = {p_spearman:.4e}")
print(f"     {'✅ Significativa' if p_spearman < 0.05 else '⚠️ Não significativa'} (α = 0.05)")
print()

direcao = "positiva" if r_pearson > 0 else "negativa"
forca = interpretar_correlacao(r_pearson)

print(f"   Correlacao {forca} {direcao}")
print("="*60)
print()


# matriz de frequencia
xbins = [1, 2, 3, 4, 5, 6]
ybins = [1, 2, 3, 4, 5, 6]
mat = np.zeros((len(ybins), len(xbins)), dtype=int)

for x, y in zip(base["q46_num"], base["q17_num"]):
    xi, yi = int(x) - 1, int(y) - 1
    if 0 <= xi < 6 and 0 <= yi < 6:
        mat[yi, xi] += 1

# anotacoes nos valores altos
thr = np.percentile(mat.flatten(), 75) if mat.max() > 0 else 1
annotations = []
for i_y, yv in enumerate(ybins):
    for i_x, xv in enumerate(xbins):
        val = mat[i_y, i_x]
        if val >= thr and val > 0:
            annotations.append(dict(
                x=xv, y=yv, text=str(int(val)),
                showarrow=False,
                font=dict(color="white", size=14, family="Arial Black"),
                xref="x", yref="y"
            ))

# pin com valor da correlacao
annotations.append(dict(
    x=0.02,
    y=0.98,
    xref="paper",
    yref="paper",
    text=f"<b>Pearson:</b> r = {r_pearson:.3f}<br><b>Spearman:</b> ρ = {rho_spearman:.3f}",
    showarrow=False,
    font=dict(size=14, color="black"),
    align="left",
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="gray",
    borderwidth=1.5,
    borderpad=8
))

cores_quente = [
    [0.0, "#fff7bc"],
    [0.25, "#fec44f"],
    [0.5, "#fe9929"],
    [0.75, "#d95f0e"],
    [1.0, "#993404"]
]

# heatmap da internet
fig = go.Figure(data=go.Heatmap(
    z=mat,
    x=xbins,
    y=ybins,
    colorscale=cores_quente,
    zmin=float(mat.min()),
    zmax=float(mat.max() if mat.max() > 0 else 1),
    colorbar=dict(
        title=dict(
            text="Frequência",
            font=dict(size=16)
        ),
        thickness=18,
        len=0.7,
        y=0.5,
        x=1.02,
        tickfont=dict(size=14)
    ),
    hovertemplate=(
        "Internet: %{x}<br>" +
        "AVA/Moodle: %{y}<br>" +
        "Frequência: %{z}<extra></extra>"
    )
))

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Acesso à internet (nota 1–6)",
        tickmode="array",
        tickvals=xbins,
        range=[0.5, 6.5],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    yaxis=dict(
        title="AVA/Moodle (nota 1–6)",
        tickmode="array",
        tickvals=ybins,
        range=[0.5, 6.5],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    height=550,
    width=650,
    margin=dict(l=100, r=120, t=40, b=80),
    annotations=annotations
)

fig.show()

html_output_path = OUTPUTS_DIR / "html" / "heatmap_internet_ava_correlacao.html"
html_output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(html_output_path),
               config={'toImageButtonOptions': {'format': 'svg', 'width': 650, 'height': 550}})

CORRELACAO
   Variável X: Acesso a internet (q46)
   Variável Y: AVA/Moodle (q17)
   N = 808 respostas válidas

   ->> Correlação de Pearson:
     r = 0.4469
     p-valor = 6.3748e-41
     ✅ Significativa (α = 0.05)

   ->> Correlação de Spearman:
     ρ = 0.4361
     p-valor = 7.5503e-39
     ✅ Significativa (α = 0.05)

   Correlacao Correlação positiva fraca a moderada positiva



#### Heatmaps de Exemplo

In [134]:
np.random.seed(42)
n = 100

# positiva forte
x_pos = np.linspace(0, 10, n)
y_pos = 2 * x_pos + np.random.normal(0, 1, n)

# nula (r ≈ 0)
x_nula = np.random.uniform(0, 10, n)
y_nula = np.random.uniform(0, 20, n)

# negativa forte (r ≈ -0.9)
x_neg = np.linspace(0, 10, n)
y_neg = 20 - 2 * x_neg + np.random.normal(0, 1, n)

# calcula Pearson
r_pos, _ = stats.pearsonr(x_pos, y_pos)
r_nula, _ = stats.pearsonr(x_nula, y_nula)
r_neg, _ = stats.pearsonr(x_neg, y_neg)

# figura 2x2
fig = plt.figure(figsize=(12, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

axes = [ax1, ax2, ax3]

# subplot: corelcao positiva
axes[0].scatter(x_pos, y_pos, alpha=0.6, s=50, color='#2ecc71', edgecolors='black', linewidth=0.5)
z = np.polyfit(x_pos, y_pos, 1)
p = np.poly1d(z)
axes[0].plot(x_pos, p(x_pos), "r--", linewidth=2, alpha=0.8, label='Reta de regressão')
axes[0].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[0].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[0].text(0.95, 0.05, f'r = {r_pos:.3f}', transform=axes[0].transAxes,
             fontsize=12, weight='bold', ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].legend(fontsize=10, loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(labelsize=11)

# subplot: correlcao negativa
axes[1].scatter(x_neg, y_neg, alpha=0.6, s=50, color='#e74c3c', edgecolors='black', linewidth=0.5)
z = np.polyfit(x_neg, y_neg, 1)
p = np.poly1d(z)
axes[1].plot(x_neg, p(x_neg), "r--", linewidth=2, alpha=0.8, label='Reta de regressão')
axes[1].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[1].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[1].text(0.95, 0.05, f'r = {r_neg:.3f}', transform=axes[1].transAxes,
             fontsize=12, weight='bold', ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(labelsize=11)

# subplot: correlacao neutra
pos3 = axes[2].get_position()
axes[2].set_position([pos3.x0 + pos3.width * 0.25, pos3.y0, pos3.width * 0.5, pos3.height])

axes[2].scatter(x_nula, y_nula, alpha=0.6, s=50, color='#95a5a6', edgecolors='black', linewidth=0.5)
axes[2].axhline(y=np.mean(y_nula), color='r', linestyle='--', linewidth=2, alpha=0.8, label='Média de Y')
axes[2].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[2].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[2].text(0.95, 0.05, f'r = {r_nula:.3f}', transform=axes[2].transAxes,
             fontsize=12, weight='bold', ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[2].legend(fontsize=10, loc='upper left')
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(labelsize=11)

plt.savefig(FIGURAS_DIR / 'correlacao_pearson_ilustrativa.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✅ Figura 1 salva: correlacao_pearson_ilustrativa.png")
plt.close()

# =============
# CORRELAÇÃO DE SPEARMAN
# =============

np.random.seed(123)
n = 150

# monotonica não-linear (exponencial)
x_mono_pos = np.linspace(0, 5, n)
y_mono_pos = np.exp(0.5 * x_mono_pos) + np.random.normal(0, 2, n)

# não monotônica (parábola - Pearson falha aqui) -> Buscar esse entendimento
x_parab = np.linspace(-3, 3, n)
y_parab = x_parab**2 + np.random.normal(0, 1, n)

# monotonica decrescente nao-linear
x_mono_neg = np.linspace(0, 5, n)
y_mono_neg = 30 / (1 + np.exp(x_mono_neg)) + np.random.normal(0, 1, n)

# Calcular correlações
r_pearson_pos, _ = stats.pearsonr(x_mono_pos, y_mono_pos)
rho_spearman_pos, _ = stats.spearmanr(x_mono_pos, y_mono_pos)

r_pearson_parab, _ = stats.pearsonr(x_parab, y_parab)
rho_spearman_parab, _ = stats.spearmanr(x_parab, y_parab)

r_pearson_neg, _ = stats.pearsonr(x_mono_neg, y_mono_neg)
rho_spearman_neg, _ = stats.spearmanr(x_mono_neg, y_mono_neg)

fig = plt.figure(figsize=(12, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

axes = [ax1, ax2, ax3]

# crescente não-linear
axes[0].scatter(x_mono_pos, y_mono_pos, alpha=0.6, s=50, color='#3498db', edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[0].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[0].text(0.05, 0.95, f'Pearson: {r_pearson_pos:.3f}\nSpearman: {rho_spearman_pos:.3f}',
             transform=axes[0].transAxes, fontsize=11, weight='bold',
             ha='left', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(labelsize=11)

# monotônica decrescente não-linear
axes[1].scatter(x_mono_neg, y_mono_neg, alpha=0.6, s=50, color='#e67e22', edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[1].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[1].text(0.05, 0.95, f'Pearson: {r_pearson_neg:.3f}\nSpearman: {rho_spearman_neg:.3f}',
             transform=axes[1].transAxes, fontsize=11, weight='bold',
             ha='left', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(labelsize=11)

# não monotônica (U invertido)
pos3 = axes[2].get_position()
axes[2].set_position([pos3.x0 + pos3.width * 0.25, pos3.y0, pos3.width * 0.5, pos3.height])

axes[2].scatter(x_parab, y_parab, alpha=0.6, s=50, color='#9b59b6', edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Variável X', fontsize=13, weight='bold')
axes[2].set_ylabel('Variável Y', fontsize=13, weight='bold')
axes[2].text(0.05, 0.95, f'Pearson: {r_pearson_parab:.3f}\nSpearman: {rho_spearman_parab:.3f}',
             transform=axes[2].transAxes, fontsize=11, weight='bold',
             ha='left', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(labelsize=11)

plt.savefig(FIGURAS_DIR / 'correlacao_spearman_ilustrativa.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✅ Figura 2 salva: correlacao_spearman_ilustrativa.png")
plt.close()

# =============
# DADOS REAIS
# ============

questoes_selecionadas_map = {
    'q43': 'Salas de aula',
    'q44': 'Laboratórios',
    'q45': 'Lab. informática',
    'q46': 'Internet',
    'q50': 'Biblioteca',
    'q51': 'Auditórios',
    'q52': 'Sanitários',
    'q56': 'Limpeza',
    'q57': 'Segurança',
    'q58': 'Acessibilidade',
    'q24': 'Portal UFT',
    'q26': 'Redes sociais',
    'q27': 'Comunic. interna',
    'q33': 'Direção Campus',
    'q35': 'Transparência',
    'q36': 'Recursos financ.'
}

colunas_para_correlacao = [f'{q}_num' for q in questoes_selecionadas_map.keys() if f'{q}_num' in df.columns]
df_correlacao = df[colunas_para_correlacao].copy()
df_correlacao.columns = [questoes_selecionadas_map[q.replace('_num', '')] for q in colunas_para_correlacao]

df_correlacao = df_correlacao.dropna(thresh=len(df_correlacao.columns) * 0.5)

# matrizes de correlação
corr_pearson = df_correlacao.corr(method='pearson')
corr_spearman = df_correlacao.corr(method='spearman')

# ================
# HEATMAP PEARSON
# =================

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
sns.heatmap(corr_pearson, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 9})
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(FIGURAS_DIR / 'heatmap_pearson_cpa.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============
# HEATMAP SPEARMAN
# =============

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool), k=1)
sns.heatmap(corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 9})
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(FIGURAS_DIR / 'heatmap_spearman_cpa.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ======================
# CORRELAÇÕES MAIS FORTES
# ======================

print("\n" + "="*70)
print("TOP 5 CORRELAÇÕES POSITIVAS MAIS FORTES (Pearson)")
print("="*70)

corr_flat = corr_pearson.where(np.tril(np.ones(corr_pearson.shape), k=-1).astype(bool))
corr_flat = corr_flat.stack().sort_values(ascending=False)

for i, (vars, corr) in enumerate(corr_flat.head(5).items(), 1):
    print(f"{i}. {vars[0]} ↔ {vars[1]}: r = {corr:.3f}")

print("\n" + "="*70)
print("TOP 5 CORRELAÇÕES POSITIVAS MAIS FORTES (Spearman)")
print("="*70)

corr_flat_spearman = corr_spearman.where(np.tril(np.ones(corr_spearman.shape), k=-1).astype(bool))
corr_flat_spearman = corr_flat_spearman.stack().sort_values(ascending=False)

for i, (vars, corr) in enumerate(corr_flat_spearman.head(5).items(), 1):
    print(f"{i}. {vars[0]} ↔ {vars[1]}: ρ = {corr:.3f}")

✅ Figura 1 salva: correlacao_pearson_ilustrativa.png
✅ Figura 2 salva: correlacao_spearman_ilustrativa.png

TOP 5 CORRELAÇÕES POSITIVAS MAIS FORTES (Pearson)
1. Lab. informática ↔ Laboratórios: r = 0.833
2. Laboratórios ↔ Salas de aula: r = 0.784
3. Recursos financ. ↔ Transparência: r = 0.750
4. Recursos financ. ↔ Direção Campus: r = 0.719
5. Recursos financ. ↔ Laboratórios: r = 0.702

TOP 5 CORRELAÇÕES POSITIVAS MAIS FORTES (Spearman)
1. Lab. informática ↔ Laboratórios: ρ = 0.832
2. Laboratórios ↔ Salas de aula: ρ = 0.784
3. Recursos financ. ↔ Transparência: ρ = 0.755
4. Recursos financ. ↔ Direção Campus: ρ = 0.712
5. Recursos financ. ↔ Laboratórios: ρ = 0.702


## 5.4. Histogramas

In [135]:
WIDTH = 800
HEIGHT = 500

x = df["q17_num"].dropna()
fig_hist = go.Figure()
fig_hist.add_histogram(
    x=x,
    xbins=dict(start=0.5, end=6.5, size=1),
    marker=dict(color="#4cc9f0"),
    name="Frequência"
)
fig_hist.update_layout(
    template="plotly_white",
    title="Distribuição de notas – AVA/Moodle (Q17)",
    xaxis=dict(
        title="Nota (1–6)",
        tickmode="array",
        tickvals=[1,2,3,4,5,6],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    yaxis=dict(
        title="Contagem",
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    margin=dict(l=80, r=40, t=70, b=70),
    width=WIDTH,
    height=HEIGHT
)

fig_hist.show()


In [136]:
questoes_infra_num = [f"{q}_num" for q in EIXOS_SINAES["Eixo 5"]["questoes"] if f"{q}_num" in df.columns]

df["media_infra"] = df[questoes_infra_num].mean(axis=1)

dados_validos = df["media_infra"].dropna()

fig = go.Figure()

fig.add_histogram(
    x=dados_validos,
    nbinsx=20,
    marker=dict(
        color='#4cc9f0',
        line=dict(color='#023e8a', width=1.5)
    ),
    hovertemplate="Média: %{x:.2f}<br>Frequência: %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Média de avaliação (escala 1-6)",
        range=[1, 6],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    yaxis=dict(
        title="Frequência",
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    margin=dict(l=80, r=40, t=70, b=80),
    height=HEIGHT,
    width=WIDTH,
    showlegend=False
)

fig.show()

try:
    fig.write_html("/content/outputs/html/histograma_media_infraestrutura.html",
               config={'toImageButtonOptions': {'format': 'svg', 'width': 800, 'height': 500}})
except ValueError as e:
    print(f":/")


## 5.5. Gráfico de Linhas

In [137]:
eixo_name_to_display = {
    EIXOS_SINAES["Eixo 1"]["nome"]: "Planejamento e\nAvaliação",
    EIXOS_SINAES["Eixo 2"]["nome"]: "Desenvolvimento\nInstitucional",
    EIXOS_SINAES["Eixo 3"]["nome"]: "Políticas\nAcadêmicas",
    EIXOS_SINAES["Eixo 4"]["nome"]: "Políticas de\nGestão",
    EIXOS_SINAES["Eixo 5"]["nome"]: "Infraestrutura",
}

ordem_eixos_display = list(eixo_name_to_display.values())

registros = []
segments_to_iterate = [seg for seg in ["Discente", "Docente", "Técnico", "Egresso"] if seg in df[coluna_segmento].unique()]

for seg in segments_to_iterate:
    dseg = df[df[coluna_segmento] == seg]
    for eixo_key, eixo_info in EIXOS_SINAES.items():
        questoes_eixo_num = [f"{q}_num" for q in eixo_info['questoes'] if f"{q}_num" in df.columns]

        if not questoes_eixo_num:
            media = np.nan
        else:
            vals = dseg[questoes_eixo_num].values.flatten()
            vals = vals[~np.isnan(vals)]
            media = np.nanmean(vals) if len(vals) > 0 else np.nan
        registros.append({"Segmento": seg, "Eixo": eixo_name_to_display[eixo_info['nome']], "Média": media})

res = pd.DataFrame(registros)

res["Eixo"] = pd.Categorical(res["Eixo"], categories=ordem_eixos_display, ordered=True)
res = res.sort_values(["Segmento", "Eixo"])

fig = go.Figure()

for seg in segments_to_iterate:
    d = res[res["Segmento"] == seg]
    fig.add_trace(go.Scatter(
        x=d["Eixo"].astype(str),
        y=d["Média"],
        mode="lines+markers",
        name=seg,
        line=dict(color=CORES_SEGMENTO.get(seg, '#888888'), width=3),
        marker=dict(size=10),
        connectgaps=False,
        hovertemplate="%{x}<br>" + seg + ": %{y:.2f}<extra></extra>"
    ))

fig.update_layout(
    template="plotly_white",
    yaxis=dict(
        title="Nota média (1–6)",
        range=[1, 6],
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5, 6],
        titlefont=dict(size=18),
        tickfont=dict(size=16)
    ),
    xaxis=dict(
        title="Eixo avaliativo",
        categoryorder="array",
        categoryarray=ordem_eixos_display,
        titlefont=dict(size=18),
        tickfont=dict(size=14)
    ),
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.02,
        font=dict(size=15),
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="gray",
        borderwidth=1.5
    ),
    margin=dict(l=80, r=150, t=40, b=100),
    height=550,
    width=950
)

fig.show()

html_output_path = OUTPUTS_DIR / "html" / "grafico_linha_eixos.html"
html_output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(html_output_path),
               config={'toImageButtonOptions': {'format': 'svg', 'width': 950, 'height': 550}})

print("salvo!")

salvo!


## 5.6. Escala Likert

In [138]:
base = df[["segmento", "q17", "q50"]].copy()

base.rename(columns={'segmento': 'grupo'}, inplace=True)

def likert_numeric_to_str_nso(val):
    if pd.isna(val):
        return "NSO"
    try:
        return str(int(val))
    except ValueError:
        return "NSO"

base.loc[:, "q17_norm"] = base["q17"].apply(to_likert_numeric).apply(likert_numeric_to_str_nso)
base.loc[:, "q50_norm"] = base["q50"].apply(to_likert_numeric).apply(likert_numeric_to_str_nso)

cores_likert = {
    "1": "#ff6b6b",
    "2": "#ffa07a",
    "3": "#ffd166",
    "4": "#06d6a0",
    "5": "#1b9aaa",
    "6": "#118ab2",
    "NSO": "#adb5bd"
}

ordem_notas = ["1", "2", "3", "4", "5", "6", "NSO"]
ordem_segmentos = [seg for seg in ["Discente", "Docente", "Técnico", "Egresso"] if seg in base["grupo"].unique()]

def criar_likert_divergente(base_dados, coluna_questao, titulo_grafico=""):
    dados_validos = base_dados[["grupo", coluna_questao]].dropna()

    contagem = dados_validos.groupby(["grupo", coluna_questao]).size().reset_index(name="n")
    total = dados_validos.groupby("grupo").size().reset_index(name="total")
    resultado = contagem.merge(total)
    resultado["perc"] = (resultado["n"] / resultado["total"]) * 100

    grid_completo = pd.DataFrame(
        [(s, n) for s in ordem_segmentos for n in ordem_notas],
        columns=["grupo", coluna_questao]
    )

    dados_finais = grid_completo.merge(resultado, on=["grupo", coluna_questao], how="left").fillna({"perc": 0})

    dados_finais["x"] = dados_finais["perc"]
    dados_finais.loc[dados_finais[coluna_questao].isin(["1", "2", "3"]), "x"] *= -1

    fig = go.Figure()

    for nota in ["1", "2", "3", "4", "5", "6"]:
        subset = dados_finais[dados_finais[coluna_questao] == nota]
        fig.add_bar(
            x=subset["x"],
            y=subset["grupo"],
            name=f"Nota {nota}",
            orientation="h",
            marker_color=cores_likert[nota],
            hovertemplate="%{y}<br>Nota " + nota + ": %{customdata:.1f}%<extra></extra>",
            customdata=subset["perc"].abs()
        )

    nso_dados = dados_finais[dados_finais[coluna_questao] == "NSO"]
    annotations = []
    for _, row in nso_dados.iterrows():
        if row["perc"] > 0:
            annotations.append(dict(
                x=0,
                y=row["grupo"],
                text=f"NSO: {row['perc']:.1f}%",
                showarrow=False,
                xanchor="center",
                font=dict(color="#495057", size=12, weight="bold")
            ))

    fig.update_layout(
        barmode="relative",
        template="plotly_white",
        title=titulo_grafico,
        xaxis=dict(
            title="Percentual de respostas",
            ticksuffix="%",
            zeroline=True,
            zerolinewidth=2,
            zerolinecolor="#333",
            titlefont=dict(size=18),
            tickfont=dict(size=16)
        ),
        yaxis=dict(
            title="Segmento",
            categoryorder="array",
            categoryarray=ordem_segmentos,
            titlefont=dict(size=18),
            tickfont=dict(size=16)
        ),
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.02,
            font=dict(size=14),
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="gray",
            borderwidth=1.5
        ),
        annotations=annotations,
        margin=dict(l=80, r=180, t=40, b=80),
        height=500,
        width=900
    )

    return fig


fig_q17 = criar_likert_divergente(base, "q17_norm", titulo_grafico="Questão 17: Qualidade do AVA/Moodle")
fig_q17_path = OUTPUTS_DIR / "html" / "likert_q17_ava.html"
fig_q17.write_html(str(fig_q17_path),
                   config={'toImageButtonOptions': {'format': 'svg', 'width': 900, 'height': 500}})

fig_q50 = criar_likert_divergente(base, "q50_norm", titulo_grafico="Questão 50: Qualidade da Biblioteca")
fig_q50_path = OUTPUTS_DIR / "html" / "likert_q50_biblioteca.html"
fig_q50.write_html(str(fig_q50_path),
                   config={'toImageButtonOptions': {'format': 'svg', 'width': 900, 'height': 500}})

fig_q17.show()
fig_q50.show()

print("ok")

ok
